# 06 — HEMT-CLIP Demo

Interactive demo of the headline **HEMT-CLIP** — the α-gated cross-attention model (`gated_fusion`), the best detector in our ablation (test F1 0.839 / AUC 0.912; notebook 04).

Given a social-media post (**title + image**), the model predicts **REAL vs FAKE** with a confidence score and explains itself with:
- the **CLIP text-image alignment α** (how well the title matches the picture in CLIP space), and
- a **cross-attention heatmap** showing where on the image the text-conditioned model looked.

The `analyze(title, image)` helper at the end works on the built-in test samples or on your own headline + image.

## Setup
Prepares the environment and loads the dataset. Idempotent — safe to re-run. Warnings are silenced so the demo output stays clean.

In [ ]:
# Bootstrap — idempotent. Safe to re-run on a fresh or warm Colab runtime.
import os, sys, subprocess, shutil

# Env vars FIRST — must be set before any transformers import.
os.environ['USE_FLAX'] = 'FALSE'
os.environ['USE_TF'] = 'FALSE'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

# Silence the deprecation/user warnings so the demo output stays clean.
import warnings
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
REPO_URL = 'https://github.com/staharizvi/hemt-clip-fnd.git'
REPO_DIR = '/content/hemt-clip-fnd'
H5_DRIVE = '/content/drive/MyDrive/hemt-clip-fnd/data/fakeddit.h5'
H5_LOCAL = '/content/fakeddit.h5'

if IN_COLAB:
    if not os.path.ismount('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
    else:
        print('Drive already mounted.')

    if os.path.exists(os.path.join(REPO_DIR, '.git')):
        print('Repo present \u2014 pulling latest\u2026')
        subprocess.run(['git', '-C', REPO_DIR, 'pull', '--quiet'], check=True)
    else:
        print('Cloning repo\u2026')
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)

    subprocess.run(['pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements.txt'], check=True)
    # jaxlib/jax/flax force numpy>=2 and clash with the project numpy pin.
    subprocess.run(['pip', 'uninstall', '-y', '-q', 'jax', 'jaxlib', 'flax'], check=False)

    if not os.path.exists(H5_LOCAL):
        if os.path.exists(H5_DRIVE):
            print(f'Copying {H5_DRIVE} -> {H5_LOCAL}\u2026')
            shutil.copy(H5_DRIVE, H5_LOCAL)
        else:
            print(f'WARNING: {H5_DRIVE} not found \u2014 run the data-prep notebook first.')
    else:
        print(f'h5 already at {H5_LOCAL}.')

    os.chdir(REPO_DIR)

print('\ncwd:', os.getcwd())
print('h5 :', H5_LOCAL, 'exists:', os.path.exists(H5_LOCAL))

## Dataset path
Point the demo at the local copy of the dataset.

In [ ]:
import yaml, pathlib
cfg_path = pathlib.Path('configs/base.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
if cfg['data']['hdf5_path'] != '/content/fakeddit.h5':
    cfg['data']['hdf5_path'] = '/content/fakeddit.h5'
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print('Patched cfg.data.hdf5_path -> /content/fakeddit.h5')
else:
    print('cfg already points at local HDF5.')

## Load the headline HEMT-CLIP
Auto-discover the latest `gated_fusion` checkpoint, build the model from `configs/base.yaml` (CLIP ViT-B/16 backbone), and load the RoBERTa tokenizer and CLIP (used to compute α).

In [ ]:
import yaml, torch
from pathlib import Path
from transformers import AutoTokenizer, CLIPModel, CLIPTokenizer
from models.hemt_clip import build_from_config
from training.evaluate import discover_checkpoints

# The headline HEMT-CLIP is the alpha-gated cross-attention variant.
VARIANT = 'gated_fusion'

with open('configs/base.yaml') as f:
    cfg = yaml.safe_load(f)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpts = discover_checkpoints(Path(cfg['checkpointing']['dir']))
if VARIANT not in ckpts:
    raise FileNotFoundError(
        f"No '{VARIANT}' checkpoint found. Train it in notebook 03 first, "
        f"or set VARIANT to one of: {sorted(ckpts)}")
ckpt_path = ckpts[VARIANT]
print('checkpoint :', ckpt_path.name)
print('device     :', device.type)

model = build_from_config(cfg, variant=VARIANT).to(device).eval()
payload = torch.load(ckpt_path, map_location=device, weights_only=False)
state = payload['model'] if isinstance(payload, dict) and 'model' in payload else payload
model.load_state_dict(state, strict=True)

tokenizer = AutoTokenizer.from_pretrained(cfg['model']['text']['name'])
clip_model = CLIPModel.from_pretrained(cfg['model']['image']['name']).to(device).eval()
clip_tokenizer = CLIPTokenizer.from_pretrained(cfg['model']['image']['name'])
print('HEMT-CLIP loaded:', VARIANT)

## Inference helpers
Self-contained inference utilities: preprocess an image, compute α via CLIP, run one HEMT-CLIP forward pass for the prediction + cross-attention weights, and render a result card. The single entry point is **`analyze(title, image, truth=None)`**.

In [ ]:
import json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from data.dataset import CLIP_MEAN, CLIP_STD


def preprocess_image(pil_img):
    img = pil_img.convert('RGB').resize((224, 224), Image.BICUBIC)
    arr = (np.asarray(img, dtype=np.float32) / 255.0).transpose(2, 0, 1)
    return (arr - CLIP_MEAN[:, None, None]) / CLIP_STD[:, None, None]


@torch.no_grad()
def compute_alpha(pil_img, text):
    """CLIP text-image cosine similarity (the alpha signal)."""
    pixel_values = torch.from_numpy(preprocess_image(pil_img)).unsqueeze(0).to(device)
    enc = clip_tokenizer([text], padding='max_length', truncation=True,
                         max_length=77, return_tensors='pt').to(device)
    img_emb = F.normalize(clip_model.get_image_features(pixel_values=pixel_values).float(), dim=-1)
    txt_emb = F.normalize(clip_model.get_text_features(
        input_ids=enc['input_ids'], attention_mask=enc['attention_mask']).float(), dim=-1)
    return float((img_emb * txt_emb).sum(dim=-1).item())


@torch.no_grad()
def predict(pil_img, text, alpha, max_text_len=128):
    """Returns (probs[2], attn_grid[P, P] or None) from one forward pass."""
    pixel_values = torch.from_numpy(preprocess_image(pil_img)).unsqueeze(0).to(device)
    enc = tokenizer([text], padding='max_length', truncation=True,
                    max_length=max_text_len, return_tensors='pt').to(device)
    batch = {
        'input_ids':      enc['input_ids'],
        'attention_mask': enc['attention_mask'],
        'pixel_values':   pixel_values,
        'alpha':          torch.tensor([alpha], dtype=torch.float32, device=device),
        'label':          torch.tensor([0], dtype=torch.long, device=device),
    }
    out = model(batch)
    logits = out['logits'].float().cpu().numpy()[0]
    probs = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()
    attn_grid = None
    if 'attention_weights' in out:
        a = out['attention_weights'].float().cpu().numpy()[0].mean(axis=0).squeeze(0)
        p = int(round(np.sqrt(len(a))))
        attn_grid = a.reshape(p, p)
    return probs, attn_grid


def interpret_alpha(alpha):
    if alpha < 0.15:
        return f'low ({alpha:.3f}) - text and image weakly aligned in CLIP space'
    if alpha < 0.30:
        return f'moderate ({alpha:.3f}) - typical band for Fakeddit headline + thumbnail pairs'
    return f'high ({alpha:.3f}) - strong CLIP text-image alignment'


def _attention_overlay(pil_img, attn_grid):
    img_arr = np.array(pil_img.convert('RGB').resize((224, 224)))
    t = torch.from_numpy(attn_grid).float().unsqueeze(0).unsqueeze(0)
    up = F.interpolate(t, size=(224, 224), mode='bilinear', align_corners=False).squeeze().numpy()
    up = (up - up.min()) / (up.max() - up.min() + 1e-8)
    return img_arr, up


def analyze(text, image, truth=None):
    """Run the headline HEMT-CLIP on one (title, image) pair and render a result card."""
    if isinstance(image, (str, Path)):
        image = Image.open(image)

    alpha = compute_alpha(image, text)
    probs, attn_grid = predict(image, text, alpha)
    pred = int(probs.argmax())
    pred_name = 'FAKE' if pred == 1 else 'REAL'
    conf = float(probs[pred])

    ncols = 3 if attn_grid is not None else 2
    fig, axes = plt.subplots(1, ncols, figsize=(4.2 * ncols, 4.2))

    axes[0].imshow(image.convert('RGB').resize((224, 224)))
    axes[0].set_title('Input image'); axes[0].axis('off')

    if attn_grid is not None:
        img_arr, up = _attention_overlay(image, attn_grid)
        axes[1].imshow(img_arr); axes[1].imshow(up, cmap='hot', alpha=0.5)
        axes[1].set_title('Cross-attention (hotter = higher)'); axes[1].axis('off')
        bar_ax = axes[2]
    else:
        bar_ax = axes[1]

    bar_ax.barh(['REAL', 'FAKE'], [probs[0], probs[1]], color=['#2e7d32', '#c62828'])
    bar_ax.set_xlim(0, 1); bar_ax.set_title('Class probability')
    bar_ax.invert_yaxis()
    for i, v in enumerate([probs[0], probs[1]]):
        bar_ax.text(min(v + 0.02, 0.85), i, f'{v:.1%}', va='center')

    verdict = f'Prediction: {pred_name}  (confidence {conf:.1%})    |    alpha = {interpret_alpha(alpha)}'
    if truth is not None:
        verdict += f'\nGround truth: {truth}  ({"correct" if truth == pred_name else "WRONG"})'
    fig.suptitle(verdict, fontsize=12)
    plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.show()

    print('Title:', text)
    return {'pred': pred_name, 'confidence': conf, 'alpha': alpha}

## Worked examples - test samples
Load a couple of REAL and FAKE posts from the held-out test split and run them through `analyze`. Each card shows the input image, the cross-attention heatmap, the class probabilities, the predicted label with confidence, the alpha alignment, and the ground-truth check.

In [ ]:
import h5py
import numpy as np
from PIL import Image


def load_test_samples(hdf5_path, n_per_class=2, seed=0):
    """A few REAL and FAKE examples from the held-out test split."""
    out = []
    with h5py.File(hdf5_path, 'r') as f:
        splits = f['splits'][:].astype(str)
        labels = f['labels'][:]
        test_mask = splits == 'test'
        rng = np.random.default_rng(seed)
        for label_val, name in [(0, 'REAL'), (1, 'FAKE')]:
            pool = np.where(test_mask & (labels == label_val))[0]
            for idx in rng.choice(pool, size=min(n_per_class, len(pool)), replace=False):
                img = f['images'][idx].transpose(1, 2, 0).astype(np.uint8)
                text = f['texts'][idx]
                text = text.decode('utf-8') if isinstance(text, bytes) else str(text)
                out.append({'idx': int(idx), 'image': Image.fromarray(img),
                            'text': text, 'truth': name})
    return out


samples = load_test_samples(cfg['data']['hdf5_path'])
for s in samples:
    print('=' * 90)
    analyze(s['text'], s['image'], truth=s['truth'])

## Try your own input
Paste any headline and point at an image to get a live prediction. This is the main cell to demo live — swap in a headline and an uploaded image and re-run.

In [ ]:
# Try your own example.
#   - Replace `my_title` with any headline.
#   - For `my_image`, point at an uploaded file:  Image.open('/content/my_photo.jpg')
#     (use the Colab file browser on the left to upload), or reuse a loaded test sample.
my_title = samples[0]['text']
my_image = samples[0]['image']

analyze(my_title, my_image)

## Where to go next
- **Full metrics & ablation:** `notebooks/04_evaluation.ipynb` (held-out test table, confusion matrices, ROC).
- **Explainability:** `notebooks/05_explainability.ipynb` (cross-attention heatmaps + SHAP + LIME, Chapter 6.4).

The cross-attention heatmap shown above is *intrinsic* to this architecture - a capability the plain-concatenation baseline cannot provide, on top of HEMT-CLIP being the best detector on test.